# 🛡️ Deepfake Detection — GPU Training Pipeline
### End-to-End Fine-Tuning of EfficientNet-B4 with PyTorch & GradCAM

This notebook trains a deepfake detector on GPU (Kaggle T4 / Google Colab T4/V100/A100).

**Workflow:**
1. Setup environment & GPU verification
2. Download dataset (Celeb-DF / DFDC / Kaggle Deepfake Dataset)
3. Extract and align face crops
4. Train EfficientNet with Warm-up + Cosine Annealing Learning Rate
5. Evaluate AUC-ROC, F1-Score, Confusion Matrix
6. Generate GradCAM explainability maps
7. Export `best_model.pth` for the Streamlit web app

In [ ]:
# Step 1: Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    print('⚠️ Running on CPU — GPU recommended for training.')

In [ ]:
# Step 2: Install dependencies
!pip install -q albumentations torchvision timm scikit-learn opencv-python matplotlib seaborn tqdm

In [ ]:
# Step 3: Clone or load repository structure
!git clone https://github.com/Gautam-Desk/DS-project.git
%cd DS-project

In [ ]:
# Step 4: Generate sample dataset or connect Kaggle dataset
# If using Kaggle dataset, point data/splits to your dataset path
from src.generate_sample_data import build_sample_dataset
build_sample_dataset(count_per_class=60)

In [ ]:
# Step 5: Start Training
from src.train import train
history = train(config_path='config.yaml')

In [ ]:
# Step 6: Full Evaluation & Metrics
from src.evaluate import evaluate
metrics = evaluate(model_path='models/best_model.pth', split='test')

In [ ]:
# Step 7: Test GradCAM Explainability on a sample
from PIL import Image
from src.predict import DeepfakePredictor
import matplotlib.pyplot as plt

predictor = DeepfakePredictor(model_path='models/best_model.pth')
sample_img = Image.open('data/splits/test/fake/sample_001.jpg')
res = predictor.predict_image(sample_img)

print(f"Prediction: {res['emoji']} {res['label']} (Prob: {res['probability']*100:.2f}%)")
if res['gradcam_pil']:
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1); plt.imshow(sample_img); plt.title('Original'); plt.axis('off')
    plt.subplot(1, 2, 2); plt.imshow(res['gradcam_pil']); plt.title('GradCAM Heatmap'); plt.axis('off')
    plt.show()